In [9]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets
from torchvision.transforms import v2
import torchvision.transforms.functional as FF
import torch.nn.functional as F
import random

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

learning_rate = 1e-3
epochs = 5

Using mps device


In [11]:
training_data = datasets.VOCDetection(
    root="data",
    year="2007",
    image_set="trainval",
    download=True,
    transform=v2.Compose([
        v2.ToImage(), v2.ToDtype(torch.float32, scale=True)
    ])
)

100.0%


In [12]:
test_data = datasets.VOCDetection(
    root="data",
    year="2007",
    image_set="test",
    download=True,
    transform=v2.Compose([
        v2.ToImage(), v2.ToDtype(torch.float32, scale=True)
    ])
)

100.0%


In [ ]:
training_dataloader = DataLoader(training_data, batch_size=32)
test_dataloader = DataLoader(test_data, batch_size=32)

In [2]:
class LateFusionModel(nn.Module):
    def __intit__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 96, 11, 4)
        self.pool1 = nn.MaxPool2d(3, 2)
        self.bn1 = nn.BatchNorm2d()
        self.conv2 = nn.Conv2d(96, 384, 5, 2, padding=2)
        self.pool2 = nn.MaxPool2d(3, 2)
        self.bn2 = nn.BatchNorm2d()
        self.conv3 = nn.Conv2d(384, 384, 3, 1, padding=1)
        self.bn3 = nn.BatchNorm2d()
        self.conv4 = nn.BatchNorm2d(384, 384, 3, 1, padding=1)
        self.conv5 = nn.BatchNorm2d(384, 356, 3, 1, padding=1)
        self.pool5 = nn.MaxPool2d(3, 2)
        self.fc6 = nn.Linear(256, 4096)
        self.fc7 = nn.Linear(4096*2, 4096)
        self.fc8 = nn.Linear(4096, 40966)
        self.fc9 = nn.Linear(4096,8)

    def forward(self, x1, x2):
        x1 = self.fc6(self.pool5(F.relu(self.conv5(F.relu(self.conv4(F.relu(self.conv3(
            self.bn2(self.pool2(self.bn1(self.pool1(F.relu(self.conv1(x1))))))
        ))))))))
        x2 = self.fc6(self.pool5(F.relu(self.conv5(F.relu(self.conv4(F.relu(self.conv3(
            self.bn2(self.pool2(self.bn1(self.pool1(F.relu(self.conv1(x2))))))
        ))))))))
        fusion = torch.cat((x1, x2), dim=1)
        fusion = F.relu(self.fc7(fusion))
        fusion = F.relu(self.fc8(fusion))
        logits = self.fc9(fusion)
        return logits

In [3]:
model = LateFusionModel()

In [6]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.to(device)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X = X.to(device)
        y = y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, curr = loss.item(), batch*batch_size + len(X)
            print(f"loss: {loss:>7f} [{curr:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct)::>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [7]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for t in range(epochs_pre):
    print(f"Epoch {t+1}\n-----------------------------------")
    train_loop(train_rot_dataloader, model, loss_fn, optimizer)
    test_loop(test_rot_dataloader, model, loss_fn)
print("Done!")

NameError: name 'learning_rate' is not defined